In [1]:
import africastalking
import os

# Initialize Africa's Talking in sandbox mode
africastalking.initialize(
    username='sandbox',
    api_key='atsk_61533322213af52898cc763b2b51a37b924562233cb562d1b1f818c91be2ff2f1406b1fb' 
)

sms = africastalking.SMS
voice = africastalking.Voice

print("Africa's Talking initialized successfully!")

Africa's Talking initialized successfully!


c:\Users\ELITE\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\africastalking\Whatsapp.py:34: UserWarning: Sandbox is currently not available for the Whatsapp service.
  warnings.warn(


In [2]:
# Alert message templates per crop per stress level
# English base messages
messages = {
    'high': {
        'Maize':    "DROUGHT ALERT: Severe drought risk in your area this week. Water your maize immediately or harvest ready cobs now. Contact your extension officer.",
        'Cassava':  "DROUGHT ALERT: Severe drought detected. Apply mulch around cassava stems to retain soil moisture urgently.",
        'Yam':      "DROUGHT ALERT: Severe drought risk. Cover yam mounds with mulch and water if possible.",
        'Rice':     "DROUGHT ALERT: Severe drought risk. Check irrigation channels immediately. Rice is at critical stress.",
        'Sorghum':  "DROUGHT ALERT: Severe drought detected. Sorghum is stressed. Harvest early if cobs are ready.",
        'Millet':   "DROUGHT ALERT: Severe drought risk. Millet is stressed. Water now if possible.",
        'Groundnut':"DROUGHT ALERT: Severe drought detected. Groundnut pods may fail. Water immediately if possible.",
        'Cocoa':    "DROUGHT ALERT: Severe drought risk. Apply mulch around cocoa trees and water if possible.",
        'Plantain': "DROUGHT ALERT: Severe drought detected. Water plantain stems immediately.",
        'Cowpea':   "DROUGHT ALERT: Severe drought risk. Cowpea is stressed. Water now or harvest ready pods.",
    },
    'medium': {
        'Maize':    "DROUGHT WARNING: Moderate drought risk detected. Monitor your maize closely and prepare to water.",
        'Cassava':  "DROUGHT WARNING: Moderate drought risk. Apply mulch around cassava to conserve moisture.",
        'Yam':      "DROUGHT WARNING: Moderate drought detected. Monitor yam mounds and water if possible.",
        'Rice':     "DROUGHT WARNING: Moderate drought risk. Check water levels in rice fields.",
        'Sorghum':  "DROUGHT WARNING: Moderate drought risk. Monitor sorghum closely this week.",
        'Millet':   "DROUGHT WARNING: Moderate drought risk. Monitor millet fields closely.",
        'Groundnut':"DROUGHT WARNING: Moderate drought detected. Monitor groundnut fields closely.",
        'Cocoa':    "DROUGHT WARNING: Moderate drought risk. Monitor cocoa trees and apply mulch.",
        'Plantain': "DROUGHT WARNING: Moderate drought risk. Monitor plantain stems for stress signs.",
        'Cowpea':   "DROUGHT WARNING: Moderate drought risk. Monitor cowpea fields closely.",
    }
}

# Crops grown per community
community_crops = {
    'Tamale':     ['Maize', 'Sorghum', 'Millet', 'Groundnut', 'Cowpea'],
    'Techiman':   ['Maize', 'Yam', 'Cassava', 'Plantain'],
    'Kumasi':     ['Maize', 'Cassava', 'Plantain', 'Cocoa'],
    'Ho':         ['Maize', 'Cassava', 'Rice', 'Cowpea'],
    'Bolgatanga': ['Maize', 'Sorghum', 'Millet', 'Groundnut'],
}

print("Message templates loaded!")
print(f"Crops covered: {len(messages['high'])} crop types")
print(f"Communities mapped: {len(community_crops)}")

# Test farmer registry (phone numbers for testing)
farmer_registry = [
    {"name": "Kofi Mensah",   "community": "Ho",       "phone": "+233XXXXXXXXX", "language": "Twi",     "literate": False},
    {"name": "Ama Asante",    "community": "Techiman", "phone": "+233XXXXXXXXX", "language": "Twi",     "literate": True},
    {"name": "Alhassan Baba", "community": "Tamale",   "phone": "+233XXXXXXXXX", "language": "Dagbani", "literate": False},
    {"name": "Efo Agbeko",    "community": "Ho",       "phone": "+233XXXXXXXXX", "language": "Ewe",     "literate": True},
]

print(f"\nFarmer registry: {len(farmer_registry)} test farmers")
print("Ready to send alerts!")

Message templates loaded!
Crops covered: 10 crop types
Communities mapped: 5

Farmer registry: 4 test farmers
Ready to send alerts!


In [3]:
import pandas as pd

# Load predictions
df_predictions = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/predictions.csv')

# Alert sending function
def send_sms_alert(farmer, community, ensemble_score):
    # Determine stress level
    if ensemble_score >= 0.75:
        level = 'high'
    else:
        level = 'medium'
    
    # Get crops for this community
    crops = community_crops.get(community, ['Maize'])
    
    # Build combined message
    crop_alerts = []
    for crop in crops[:2]:  # limit to 2 crops per SMS
        msg = messages[level][crop]
        crop_alerts.append(msg)
    
    full_message = f"AgroAlert Ghana | {community} | Score: {ensemble_score:.2f}\n" + \
                   "\n".join(crop_alerts)
    
    # Send SMS (sandbox mode)
    try:
        response = sms.send(full_message, [farmer['phone']])
        return {"status": "sent", "farmer": farmer['name'], 
                "community": community, "score": ensemble_score}
    except Exception as e:
        return {"status": "error", "farmer": farmer['name'], "error": str(e)}

# Get communities with alerts triggered
alerted_communities = df_predictions[df_predictions['alert_triggered'] == 1]['community'].unique()
print(f"Communities with active drought alerts: {list(alerted_communities)}")

# Send alerts to farmers in alerted communities
alert_log = []
for farmer in farmer_registry:
    if farmer['community'] in alerted_communities:
        # Get latest score for this community
        score = df_predictions[
            df_predictions['community'] == farmer['community']
        ]['ensemble_score'].max()
        
        if farmer['literate']:
            # Send SMS to literate farmers
            result = send_sms_alert(farmer, farmer['community'], score)
            alert_log.append(result)
            print(f"SMS sent to {farmer['name']} ({farmer['community']}) — Score: {score:.2f}")
        else:
            # Log voice call needed for non-literate farmers
            alert_log.append({
                "status": "voice_needed",
                "farmer": farmer['name'],
                "community": farmer['community'],
                "language": farmer['language'],
                "score": score
            })
            print(f"Voice call needed for {farmer['name']} ({farmer['language']}) — Score: {score:.2f}")

print(f"\nAlert log: {alert_log}")

Communities with active drought alerts: ['Ho', 'Kumasi', 'Techiman']
Voice call needed for Kofi Mensah (Twi) — Score: 0.88
SMS sent to Ama Asante (Techiman) — Score: 0.88
SMS sent to Efo Agbeko (Ho) — Score: 0.88

Alert log: [{'status': 'voice_needed', 'farmer': 'Kofi Mensah', 'community': 'Ho', 'language': 'Twi', 'score': np.float64(0.879011543626971)}, {'status': 'error', 'farmer': 'Ama Asante', 'error': 'Invalid phone number: +233XXXXXXXXX'}, {'status': 'error', 'farmer': 'Efo Agbeko', 'error': 'Invalid phone number: +233XXXXXXXXX'}]


In [4]:
# Save alert log to CSV
import json
import pandas as pd
from datetime import datetime

alert_df = pd.DataFrame(alert_log)
alert_df['timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

alert_df.to_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/alert_log.csv', index=False)
print("Alert log saved!")
print(alert_df)

Alert log saved!
         status       farmer community language     score  \
0  voice_needed  Kofi Mensah        Ho      Twi  0.879012   
1         error   Ama Asante       NaN      NaN       NaN   
2         error   Efo Agbeko       NaN      NaN       NaN   

                                 error            timestamp  
0                                  NaN  2026-06-27 19:33:44  
1  Invalid phone number: +233XXXXXXXXX  2026-06-27 19:33:44  
2  Invalid phone number: +233XXXXXXXXX  2026-06-27 19:33:44  


In [8]:
# Update registry with your real number for testing
farmer_registry_test = [
    {"name": "Ishmael Fynn Cudjoe",   "community": "Ho",       
     "phone": "+233201828606",  # replace with your real number
     "language": "Twi", "literate": False},
    {"name": "Anastasia Oheneba Duah",    "community": "Techiman", 
     "phone": "+233595798316",  # replace with your real number
     "language": "Twi", "literate": False},

_IncompleteInputError: incomplete input (253619123.py, line 8)

In [9]:
# Language per community
community_language = {
    'Tamale':     'Dagbani',
    'Bolgatanga': 'Dagbani',
    'Ho':         'Ewe',
    'Kumasi':     'Twi',
    'Techiman':   'Twi',
}

# Voice call messages in local languages
voice_messages = {
    'Twi':     "Merebɔ wo kra. Ɛwɔ mmoawa a ɛrebɛba wo fi. Fa wo bobe gu asase so seesei.",
    'Ewe':     "Miafia ame. Dzodzi le gbɔ. Trɔe wò ха̃ame le fifia.",
    'Dagbani': "N bɛ kuli n nyɛ i. Kpariba bɛ wari. I ni kpɛm i viɛla ziŋ."
}

MY_NUMBER = "+233201828606"  

farmer_registry_test = [
    {"name": "Kofi Mensah",   "community": "Ho",       "phone": MY_NUMBER, "literate": False},
    {"name": "Ama Asante",    "community": "Techiman", "phone": MY_NUMBER, "literate": True},
    {"name": "Alhassan Baba", "community": "Tamale",   "phone": MY_NUMBER, "literate": False},
    {"name": "Efo Agbeko",    "community": "Ho",       "phone": MY_NUMBER, "literate": True},
]

def send_alert(farmer, score):
    community = farmer['community']
    language = community_language[community]
    crops = community_crops[community]
    level = 'high' if score >= 0.75 else 'medium'

    if farmer['literate']:
        # SMS in English
        crop_list = " and ".join(crops[:2])
        msg = (f"AgroAlert Ghana | {community} | "
               f"DROUGHT ALERT: Severe drought risk detected for {crop_list} farmers. "
               f"Risk score: {score:.2f}. Take action immediately.")
        try:
            response = sms.send(msg, [farmer['phone']])
            print(f"✓ SMS sent to {farmer['name']} ({community}) in English")
        except Exception as e:
            print(f"✗ SMS error for {farmer['name']}: {e}")
    else:
        # Voice call in local language
        voice_msg = voice_messages[language]
        print(f"✓ Voice call queued for {farmer['name']} ({community}) in {language}:")
        print(f"  Message: {voice_msg}")

# Run alerts for alerted communities
alerted = df_predictions[df_predictions['alert_triggered'] == 1]['community'].unique()
print(f"Alerted communities: {list(alerted)}\n")

for farmer in farmer_registry_test:
    if farmer['community'] in alerted:
        score = df_predictions[
            df_predictions['community'] == farmer['community']
        ]['ensemble_score'].max()
        send_alert(farmer, score)

Alerted communities: ['Ho', 'Kumasi', 'Techiman']

✓ Voice call queued for Kofi Mensah (Ho) in Ewe:
  Message: Miafia ame. Dzodzi le gbɔ. Trɔe wò ха̃ame le fifia.
✗ SMS error for Ama Asante: HTTPSConnectionPool(host='api.sandbox.africastalking.com', port=443): Max retries exceeded with url: /version1/messaging (Caused by SSLError(SSLError(1, '[SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:1082)')))
✓ SMS sent to Efo Agbeko (Ho) in English


In [15]:
def make_voice_call(farmer, score):
    community = farmer['community']
    language = community_language[community]
    voice_msg = voice_messages[language]
    
    # Place voice call via Africa's Talking
    try:
        response = voice.call(
            callFrom='+233201828606',  # Africa's Talking sandbox number
            callTo=[farmer['phone']]
        )
        print(f"✓ Voice call placed to {farmer['name']} in {language}")
        print(f"  Response: {response}")
    except Exception as e:
        print(f"✗ Voice call error: {e}")

# Test voice call for illiterate farmer
kofi = {"name": "Kofi Mensah", "community": "Ho", 
        "phone": "+2330201828606",  
        "literate": False}
